# Planejamento e Reflexão

Planejar e refletir gastam a mesma moeda: mais chamadas ao modelo antes de entregar a resposta. Planejar é decidir a sequência de passos; refletir é submeter o que já saiu a uma crítica.

O notebook parte de uma tarefa que o laço não resolve, nomeia o laço que ele já usa, põe o plano dentro do estado do agente e termina comparando as duas rotas sob o mesmo orçamento.

In [ ]:
# No Google Colab, descomente e rode uma vez (Ambiente de execução > GPU).
# !pip install -q "agentkit @ git+https://github.com/silvaan/agentic-ai"

import time
from pathlib import Path
from typing import Literal

import pandas as pd
import torch
from pydantic import BaseModel, Field

from agentkit import LLM, Agent, tool

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
llm = LLM(MODEL_NAME, device=device, temperature=0.0, max_tokens=250)
print(llm.model)

## A tarefa que falha em um passo

Os três arquivos abaixo simulam relatórios mensais, cada um com um número dentro. A tarefa é somar os três, e ela só termina depois de listar a pasta, ler cada arquivo e fazer a conta.

In [ ]:
REPORTS = Path("workspace/reports")
REPORTS.mkdir(parents=True, exist_ok=True)
MONTHS = {"janeiro.txt": 1200, "fevereiro.txt": 950, "marco.txt": 1430}
for name, value in MONTHS.items():
    (REPORTS / name).write_text(f"Total de entregas: {value}\n", encoding="utf-8")
print(sorted(path.name for path in REPORTS.iterdir()), "| soma correta:", sum(MONTHS.values()))

In [ ]:
@tool
def list_files() -> str:
    """Lista os arquivos disponíveis na pasta de relatórios."""
    return ", ".join(sorted(path.name for path in REPORTS.iterdir()))


@tool
def read_file(name: str) -> str:
    """Lê um arquivo da pasta de relatórios e devolve o conteúdo."""
    return (REPORTS / name).read_text(encoding="utf-8")


TOOLS = [list_files, read_file]

In [ ]:
TASK = "Liste a pasta de relatórios e leia o total de entregas de cada arquivo."

started = time.perf_counter()
direct_messages = Agent(llm, TOOLS, max_steps=6).run(TASK)
direct_seconds = time.perf_counter() - started
for message in direct_messages:
    print(message["role"], ":", message.get("content") or message["tool_calls"])

O agente pediu duas chamadas no mesmo turno: listou a pasta e, junto, mandou ler um arquivo cujo nome ele inventou. A listagem voltou certa, a leitura falhou porque `total_deliveries.txt` não existe, e ele encerrou pedindo que o caminho fosse conferido.

O segundo argumento foi escrito antes de a primeira observação existir. É essa dependência entre passos que um plano derivado de uma observação resolve.

## ReAct: pensar, agir, observar

O laço que acabou de rodar tem nome. Ele alterna raciocínio, chamada de ferramenta e observação, e decide o passo seguinte olhando o que apareceu. Esse padrão se chama ReAct, e é o mesmo laço já usado com ferramentas. A tabela abaixo é a execução anterior, vista como sequência de eventos.

In [ ]:
pd.DataFrame([{
    "papel": message["role"],
    "chamadas": message.get("tool_calls", ""),
    "conteúdo": (message.get("content") or "")[:60],
} for message in direct_messages])

A tabela mostra o padrão: uma mensagem do assistente com as chamadas, as observações que voltaram e a mensagem final. O defeito da execução está na primeira linha, em que duas ações foram decididas de uma vez, sem esperar a observação da primeira.

### Exercício 1

Rode o mesmo laço em uma pergunta sobre um arquivo só, nomeando o arquivo no pedido, e compare a tabela com a de cima. Responda quantas voltas cada uma consumiu e em qual delas o modelo esperou a observação antes de decidir a chamada seguinte.

In [ ]:
# Seu código aqui

## Plano no estado do agente

No ReAct o próximo passo é decidido a cada volta. A alternativa é escrever a sequência inteira antes de agir e guardá-la no estado, que é um dicionário com a tarefa, o plano, as observações e o passo atual. O plano é escrito pelo modelo, e não pelo programa, mas depois de uma observação: sem saber quais arquivos existem, ele planeja no escuro.

In [ ]:
def new_state(task: str) -> dict:
    """Cria o estado da execução, com o plano e o que já foi observado."""
    return {"task": task, "plan": [], "observations": [], "step": 0}

In [ ]:
class Plan(BaseModel):
    steps: list[str] = Field(min_length=2, max_length=4)


EXAMPLE = """Exemplo para outra tarefa.
Tarefa: descobrir quantas fotos existem na pasta e abrir a mais antiga.
Passos:
1. Listar os arquivos da pasta.
2. Abrir o arquivo mais antigo da listagem."""

In [ ]:
def make_plan(task: str, tools: list, context: str = "") -> list[str]:
    """Pede ao modelo um plano de dois a quatro passos, validado pelo esquema."""
    available = "\n".join(f"- {fn.tool_schema['name']}: {fn.tool_schema['description']}" for fn in tools)
    return llm.generate_structured([{"role": "user", "content": (
        f"{EXAMPLE}\n\nAgora escreva os passos para a tarefa abaixo, em português, "
        f"cada passo em uma frase.\n\nFerramentas:\n{available}\n"
        f"{context}Tarefa: {task}"
    )}], Plan, max_tokens=200).steps

In [ ]:
seen = f"Arquivos na pasta: {list_files()}\n"
state = new_state(TASK)
state["plan"] = make_plan(TASK, TOOLS, context=seen)
pd.DataFrame({"passo": state["plan"]})

O plano existe antes de qualquer execução e pode ser lido por quem escreveu o programa, o que é a diferença prática entre plano como estrutura e plano como parágrafo. O segundo passo esconde uma repetição dentro de um passo só, porque ler cada arquivo são três chamadas, e não uma.

### O laço que avança o plano

O plano está no estado, então executá-lo é avançar um campo. Cada passo recebe as observações anteriores no contexto, e depois de cada observação o agente decide se o plano ainda serve. O replanejamento tem limite, porque um agente que sempre aceita replanejar nunca termina.

In [ ]:
class Decision(BaseModel):
    action: Literal["continuar", "replanejar"]


def decide(state: dict) -> str:
    """Pergunta ao modelo se o plano ainda serve, depois da última observação."""
    return llm.generate_structured([{"role": "user", "content": (
        f"Tarefa: {state['task']}\nPasso executado: {state['plan'][state['step'] - 1]}\n"
        f"Observação: {state['observations'][-1]}\n\nO plano ainda serve?"
    )}], Decision, max_tokens=30).action

In [ ]:
def replan(state: dict, tools: list) -> list[str]:
    """Refaz apenas os passos que faltam, a partir do que já foi observado."""
    done = "\n".join(f"- {item}" for item in state["observations"])
    return make_plan(f"{state['task']}\n\nJá observado:\n{done}\nEscreva só o que falta.", tools)

In [ ]:
def run_planned_agent(state: dict, tools: list, max_replans: int = 1) -> dict:
    """Avança o plano com as observações no contexto e replaneja no máximo uma vez."""
    replans = 0
    while state["step"] < len(state["plan"]):
        seen = "\n".join(f"- {item}" for item in state["observations"])
        instruction = state["plan"][state["step"]]
        if seen:
            instruction = f"<observações>\n{seen}\n</observações>\n\n{instruction}"
        messages = Agent(llm, tools, max_steps=5).run(instruction)
        state["observations"].append((messages[-1]["content"] or "").strip())
        state["step"] += 1
        if replans < max_replans and decide(state) == "replanejar":
            state["plan"] = state["plan"][: state["step"]] + replan(state, tools)
            replans += 1
    return state

In [ ]:
def answer_from(state: dict) -> str:
    """Responde a tarefa original com o que as observações trouxeram."""
    notes = "\n".join(f"- {item}" for item in state["observations"])
    return llm.invoke([{"role": "user", "content": (
        f"<observações>\n{notes}\n</observações>\n\n"
        f"Com base nas observações acima, responda: {state['task']}"
    )}], max_tokens=120)

In [ ]:
run_planned_agent(state, TOOLS)
print(answer_from(state))
pd.DataFrame({"passo": state["plan"][: state["step"]], "observação": state["observations"]})

O agente executou os passos, replanejou uma vez e leu um arquivo só, respondendo com o total de fevereiro. O plano estava certo em intenção e errado em granularidade: o passo que diz para cada arquivo não se desdobra sozinho em três chamadas.

### Replanejamento

Plano fixo pressupõe que o mundo não muda durante a execução. A célula seguinte remove um arquivo antes de o agente chegar ao passo que o lê.

In [ ]:
(REPORTS / "marco.txt").unlink()

stale = new_state(TASK)
stale["plan"] = make_plan(TASK, TOOLS)
run_planned_agent(stale, TOOLS)
print(stale["step"], "passos |", len(stale["plan"]), "no plano final")
pd.DataFrame({"observação": stale["observations"]})

O plano antigo continua o mesmo, porque foi montado antes da mudança, e a execução segue nele até o replanejamento. Replanejar é derivar o plano de novo a partir do estado atual, e o limite de uma vez existe porque um agente que sempre aceita replanejar nunca termina.

In [ ]:
for name, value in MONTHS.items():
    (REPORTS / name).write_text(f"Total de entregas: {value}\n", encoding="utf-8")
print(list_files())

### Exercício 2

Acrescente um arquivo novo à pasta depois de o plano ter sido montado e rode o agente. Responda se ele replanejou, e o que na observação levou a essa decisão.

In [ ]:
# Seu código aqui

## Reflexão

A segunda técnica não mexe no caminho até a resposta, e sim na resposta pronta. Pedir ao modelo que avalie um texto sem critério devolve elogio, então a crítica útil tem rubrica declarada, saída estruturada e um veredito de conjunto fechado.

In [ ]:
class Critique(BaseModel):
    score: int = Field(ge=0, le=5)
    issues: list[str]
    verdict: Literal["accept", "revise"]


def critique(task: str, answer: str) -> Critique:
    """Avalia uma resposta segundo a rubrica de precisão e completude."""
    return llm.generate_structured([{"role": "user", "content": (
        f"Tarefa: {task}\nResposta: {answer}\n\n"
        "Dê uma nota de 0 a 5 para precisão e completude, liste os problemas "
        "concretos e decida entre accept e revise."
    )}], Critique, max_tokens=250)

In [ ]:
weak_answer = "A soma das entregas dá uns três mil, mais ou menos."
review = critique(TASK, weak_answer)
print(review.score, review.verdict)
for issue in review.issues:
    print(" -", issue)

A nota e a lista de problemas são valores, e não prosa, então o programa pode decidir com eles. A crítica só melhora o que já está no contexto: uma resposta errada por falta de dado continua errada depois de qualquer número de rodadas.

### Quando a revisão não ajuda

O caso seguinte parte de uma resposta correta e simples, e a submete ao mesmo crítico.

In [ ]:
question = "Qual é a capital da Austrália? Responda apenas com a cidade."
first_answer = llm.invoke([{"role": "user", "content": question}], max_tokens=20)
second_review = critique(question, first_answer)
print(first_answer, "|", second_review.score, second_review.verdict)
for issue in second_review.issues:
    print(" -", issue)

O veredito foi de aceitação para uma resposta certa, e mesmo assim a lista de problemas veio com dois apontamentos falsos: um afirma que a Austrália tem duas capitais, e o outro diz que o texto está incorreto e que a resposta certa seria justamente a que foi dada. Um laço guiado pela lista de problemas, em vez do veredito, reescreveria uma resposta que já estava correta.

### Exercício 3

Reescreva a rubrica de `critique` para que ela exija evidência de um arquivo em cada apontamento, e rode de novo sobre a resposta da capital. Responda se o veredito mudou e se os apontamentos mudaram junto.

In [ ]:
# Seu código aqui

## Orçamento pareado

As duas rotas resolvem a mesma tarefa gastando chamadas de formas diferentes. Compará-las só é honesto sob o mesmo orçamento, e a comparação olha acerto, chamadas e tempo.

In [ ]:
started = time.perf_counter()
planned = new_state(TASK)
planned["plan"] = make_plan(TASK, TOOLS)
run_planned_agent(planned, TOOLS)
planned_answer = answer_from(planned)
planned_seconds = time.perf_counter() - started
print(planned_answer)

In [ ]:
target = str(sum(MONTHS.values()))
direct_calls = sum(1 for message in direct_messages if message["role"] == "tool")
pd.DataFrame([
    {"rota": "ReAct direto", "chamadas": direct_calls,
     "segundos": round(direct_seconds, 1),
     "acertou": target in (direct_messages[-1]["content"] or "")},
    {"rota": "plano no estado", "chamadas": planned["step"],
     "segundos": round(planned_seconds, 1), "acertou": target in planned_answer},
])

As duas rotas erram, e a do plano custa quatro vezes mais tempo para chegar ao mesmo lugar. Decompor só paga quando cada passo cabe em uma chamada, e aqui um dos passos não cabe.

O que a tabela mede não é qual técnica é melhor, e sim se este agente, com estas ferramentas e este modelo, ganha alguma coisa planejando. A resposta, neste caso, é não.

### Exercício 4

Rode as duas rotas em uma tarefa de um passo só, como ler um único arquivo. Responda qual delas ficou mais cara e o que isso diz sobre quando vale decompor.

In [ ]:
# Seu código aqui